# BICEP Basic Analysis Tutorial

This notebook demonstrates how to run BICEP analyses to estimate electrical infrastructure upgrade costs for different geographic scopes.

**Prerequisites**: 
- BICEP must be installed: From the BICEP root directory, run `pip install -e .`
- The notebook kernel should use the `bicep-env` conda environment

## Setup

First, import the BICEP analysis module:

In [1]:
# Add BICEP to Python path
import sys
from pathlib import Path

# Get the BICEP root directory (two levels up from this notebook)
bicep_root = Path.cwd().parent.parent
if str(bicep_root) not in sys.path:
    sys.path.insert(0, str(bicep_root))

print(f"BICEP root: {bicep_root}")
print("Python path updated! You can now import bicep modules.")

BICEP root: /Users/faye994/code/BICEP
Python path updated! You can now import bicep modules.


In [2]:
from bicep.analysis import BicepResults

## Example 1: All States Analysis

Load pre-computed results for the Business-As-Usual (BAU) scenario across all US states using local SQLite database:

In [ ]:
# By default, target_states='all' analyzes all US states
bau_all = BicepResults(scenario='bau', mode='local', target_states='all')

### View Total Cost

Get the total infrastructure upgrade cost across all states and years:

### Visualize Cost Drivers

Plot the breakdown of costs by different upgrade drivers (EVs, heat pumps, solar, load growth):

In [ ]:
print(f'Total BAU cost (all states): ${bau_all.total_cost:,.0f}')
print(f'Residential costs: ${bau_all.total_residential_costs:,.0f}')
print(f'Commercial costs: ${bau_all.total_commercial_costs:,.0f}')

In [ ]:
bau_all.plot_drivers()

## Example 2: Single State Analysis

Analyze a single state (California) to see state-specific infrastructure costs:

In [ ]:
# Analyze just California
bau_ca = BicepResults(scenario='bau', mode='local', target_states='CA')

### California Results

In [ ]:
print(f'Total BAU cost (California only): ${bau_ca.total_cost:,.0f}')
print(f'Residential costs: ${bau_ca.total_residential_costs:,.0f}')
print(f'Commercial costs: ${bau_ca.total_commercial_costs:,.0f}')

# View first few rows of the buildings DataFrame
print(f'\nNumber of buildings analyzed: {len(bau_ca.buildings):,}')
bau_ca.buildings.head()

In [ ]:
bau_ca.plot_drivers()

## Example 3: Multiple States Analysis

Analyze a specific set of states (California, Texas, and Washington) together:

In [ ]:
# Analyze California, Texas, and Washington together
bau_multi = BicepResults(scenario='bau', mode='local', target_states=['CA', 'TX', 'WA'])

### Multi-State Results

In [ ]:
print(f'Total BAU cost (CA, TX, WA): ${bau_multi.total_cost:,.0f}')
print(f'Residential costs: ${bau_multi.total_residential_costs:,.0f}')
print(f'Commercial costs: ${bau_multi.total_commercial_costs:,.0f}')

# Break down costs by state
print('\nCosts by state:')
state_costs = bau_multi.buildings.groupby('state')['total_upgrade_cost'].sum()
for state in ['CA', 'TX', 'WA']:
    print(f'  {state}: ${state_costs[state]:,.0f}')

In [ ]:
bau_multi.plot_drivers()

## Example 4: Comparing Scenarios

Compare the BAU scenario with the High Electrification scenario for California:

In [ ]:
# Load High Electrification scenario for California
high_ca = BicepResults(scenario='high', mode='local', target_states='CA')

### Scenario Comparison

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Scenario': ['BAU', 'High Electrification'],
    'Total Cost': [bau_ca.total_cost, high_ca.total_cost],
    'Residential Cost': [bau_ca.total_residential_costs, high_ca.total_residential_costs],
    'Commercial Cost': [bau_ca.total_commercial_costs, high_ca.total_commercial_costs]
})

# Format as currency
for col in ['Total Cost', 'Residential Cost', 'Commercial Cost']:
    comparison[col] = comparison[col].apply(lambda x: f'${x:,.0f}')

comparison

In [ ]:
# Compare visualizations side by side
print("BAU Scenario Cost Drivers:")
bau_ca.plot_drivers()

print("\nHigh Electrification Scenario Cost Drivers:")
high_ca.plot_drivers()

## Summary

This notebook demonstrated BICEP's flexible geographic analysis capabilities:

1. **All States** (`target_states='all'`): Analyze the entire United States
2. **Single State** (`target_states='CA'`): Focus on one specific state
3. **Multiple States** (`target_states=['CA', 'TX', 'WA']`): Analyze a custom set of states
4. **Scenario Comparison**: Compare BAU vs High Electrification scenarios

### Key Output Attributes

Each `BicepResults` object provides:
- `total_cost`: Total infrastructure upgrade cost across all years
- `total_residential_costs`: Residential building upgrade costs
- `total_commercial_costs`: Commercial building upgrade costs
- `buildings`: DataFrame with detailed building-level results
- `plot_drivers()`: Visualize cost breakdown by technology driver

### Available Scenarios

- `'bau'`: Business-As-Usual electrification scenario
- `'high'`: High Electrification scenario with aggressive adoption

### Database Modes

- `mode='local'`: Uses local SQLite database at `data/bicep.x-stock.db`
- `mode='pnnl'`: Uses Azure SQL Server database (requires network access)